In [1]:
pip install wikipedia pandas feedparser requests

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.3 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=2565536cd146bb0d67f08e2bc23854aa350e03c4e00d5d94c6137ada1ec23bdb
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=bc71c4917eac599e6fee54902683be613d5bd263943ccedf78b5a73daf02a928
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built wikipedia sgmllib3k


In [46]:
import wikipedia
import pandas as pd
import feedparser
import requests
import time

# -------------------------
# Initialize
# -------------------------
data = []

topics = [
    "Artificial Intelligence",
    "Cybersecurity",
    "Blockchain",
    "Cloud Computing",
    "Internet of Things",
    "Data Science",
    "Quantum Computing",
    "Computer Networks"
]

# Target number of documents per topic per type
targets = {
    "Wikipedia": 25,        # 25 per topic → 8*25=200
    "GitHub": 12,           # 12 per topic → 8*12=96
    "Blog": 10,             # 10 per topic → 8*10=80
    "StackOverflow": 12,    # 12 per topic → 8*12=96
}

# -------------------------
# 1️⃣ Wikipedia documents
# -------------------------
for topic in topics:
    results = wikipedia.search(topic, results=targets["Wikipedia"])
    for r in results:
        try:
            page = wikipedia.page(r)
            data.append({
                "title": page.title,
                "text": page.content,
                "link": page.url,
                "topic": topic,
                "type": "Wikipedia"
            })
        except:
            pass
print("Wikipedia collected")

# -------------------------
# 2️⃣ GitHub README files
# -------------------------
github_topics = {
    "artificial intelligence": "Artificial Intelligence",
    "cybersecurity": "Cybersecurity",
    "blockchain": "Blockchain",
    "cloud computing": "Cloud Computing",
    "data science": "Data Science",
    "iot": "Internet of Things",
    "network security": "Cybersecurity",
    "machine learning": "Artificial Intelligence",
    "quantum computing": "Quantum Computing",
    "computer networks": "Computer Networks"
}

for search_topic, dataset_topic in github_topics.items():
    url = f"https://api.github.com/search/repositories?q={search_topic}&sort=stars&order=desc&per_page={targets['GitHub']}"
    try:
        repos = requests.get(url).json()["items"]
        for repo in repos:
            readme_url = f"https://raw.githubusercontent.com/{repo['full_name']}/master/README.md"
            r = requests.get(readme_url)
            if r.status_code == 200:
                data.append({
                    "title": repo["name"],
                    "text": r.text,
                    "link": repo["html_url"],
                    "topic": dataset_topic,
                    "type": "GitHub"
                })
        time.sleep(1)  # avoid GitHub rate limit
    except:
        pass
print("GitHub collected")

# -------------------------
# 3️⃣ Technical Blogs (Medium)
# -------------------------
blog_feeds = {
    "https://medium.com/feed/tag/artificial-intelligence": "Artificial Intelligence",
    "https://medium.com/feed/tag/cybersecurity": "Cybersecurity",
    "https://medium.com/feed/tag/blockchain": "Blockchain",
    "https://medium.com/feed/tag/cloud-computing": "Cloud Computing",
    "https://medium.com/feed/tag/data-science": "Data Science",
    "https://medium.com/feed/tag/machine-learning": "Artificial Intelligence",
    "https://medium.com/feed/tag/devops": "Cloud Computing",
    "https://medium.com/feed/tag/kubernetes": "Cloud Computing",
    "https://medium.com/feed/tag/iot": "Internet of Things",
    "https://medium.com/feed/tag/quantum-computing": "Quantum Computing",
    "https://medium.com/feed/tag/computer-networks": "Computer Networks"
}

for feed, dataset_topic in blog_feeds.items():
    parsed = feedparser.parse(feed)
    count = 0
    for entry in parsed.entries:
        if count >= targets["Blog"]:
            break
        text = entry.get("summary", "")
        data.append({
            "title": entry.title,
            "text": text,
            "link": entry.link,
            "topic": dataset_topic,
            "type": "Blog"
        })
        count += 1
print("Blogs collected")

# -------------------------
# 4️⃣ StackOverflow Q&A
# -------------------------
stack_tags = {
    "python": "Programming",
    "machine-learning": "Artificial Intelligence",
    "cybersecurity": "Cybersecurity",
    "blockchain": "Blockchain",
    "data-science": "Data Science",
    "cloud": "Cloud Computing",
    "quantum-computing": "Quantum Computing",
    "computer-networks": "Computer Networks"
}

for tag, dataset_topic in stack_tags.items():
    stack_api = f"https://api.stackexchange.com/2.3/questions?order=desc&sort=votes&site=stackoverflow&pagesize={targets['StackOverflow']}&tagged={tag}"
    try:
        r = requests.get(stack_api).json()
        for q in r.get("items", []):
            data.append({
                "title": q["title"],
                "text": q["title"],  # short document
                "link": q["link"],
                "topic": dataset_topic,
                "type": "StackOverflow"
            })
    except:
        pass
print("StackOverflow collected")

# -------------------------
# -------------------------
# Finalize Dataset
# -------------------------
df = pd.DataFrame(data)

# Standardize topic names first
df['topic'] = df['topic'].str.title()

# Then merge duplicates / mapping
df['topic'] = df['topic'].replace({
    "Programming": "Artificial Intelligence",
    "Internet Of Things": "Internet of Things"
})

# Add document length
df['doc_length'] = df['text'].apply(lambda x: len(str(x).split()))

extra_topics = [
    "Artificial Intelligence",
    "Cybersecurity",
    "Blockchain",
    "Cloud Computing",
    "Internet of Things",
    "Data Science",
    "Quantum Computing",
    "Computer Networks"
]

for topic in extra_topics:
    results = wikipedia.search(topic, results=5)
    for r in results:
        try:
            page = wikipedia.page(r)
            df = pd.concat([df, pd.DataFrame([{
                "title": page.title,
                "text": page.content,
                "link": page.url,
                "topic": topic,
                "type": "Wikipedia",
                "doc_length": len(page.content.split())
            }])], ignore_index=True)
        except:
            pass

# Save CSV
df.to_csv("balanced_corpus_final.csv", index=False)

# Quick stats
print("Total documents:", len(df))

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Wikipedia collected
GitHub collected
Blogs collected
StackOverflow collected


/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Total documents: 506


In [47]:
df["type"].value_counts()


,count
type,
Wikipedia,204
GitHub,110
Blog,108
StackOverflow,84


In [48]:
df["topic"].value_counts()

,count
topic,
Artificial Intelligence,93
Cloud Computing,81
Cybersecurity,65
Blockchain,59
Quantum Computing,59
Data Science,58
Computer Networks,46
Internet of Things,45


In [51]:
df

,title,text,link,topic,type,doc_length
0,Artificial intelligence,Artificial intelligence (AI) is the capability...,https://en.wikipedia.org/wiki/Artificial_intel...,Artificial Intelligence,Wikipedia,12886
1,Applications of artificial intelligence,Artificial intelligence is the capability of t...,https://en.wikipedia.org/wiki/Applications_of_...,Artificial Intelligence,Wikipedia,6911
2,Artificial general intelligence,Artificial general intelligence (AGI) is a typ...,https://en.wikipedia.org/wiki/Artificial_gener...,Artificial Intelligence,Wikipedia,7321
3,History of artificial intelligence,The history of artificial intelligence (AI) be...,https://en.wikipedia.org/wiki/History_of_artif...,Artificial Intelligence,Wikipedia,14470
4,Timeline of artificial intelligence,"This is a timeline of artificial intelligence,...",https://en.wikipedia.org/wiki/Timeline_of_arti...,Artificial Intelligence,Wikipedia,967
...,...,...,...,...,...,...
501,Post-quantum cryptography,"Post-quantum cryptography (PQC), sometimes ref...",https://en.wikipedia.org/wiki/Post-quantum_cry...,Quantum Computing,Wikipedia,4562
502,Computer network engineering,Computer network engineering is a technology d...,https://en.wikipedia.org/wiki/Computer_network...,Computer Networks,Wikipedia,2251
503,Port (computer networking),"In computer networking, a port is a communicat...",https://en.wikipedia.org/wiki/Port_(computer_n...,Computer Networks,Wikipedia,1520
504,Computer network diagram,A computer network diagram is a schematic depi...,https://en.wikipedia.org/wiki/Computer_network...,Computer Networks,Wikipedia,444


In [59]:
num_rows_less_than_300 = len(df[df['doc_length'] < 300])
print(f"Number of rows with doc_length < 300: {num_rows_less_than_300}")


num_rows_greater_than_3000 = len(df[df['doc_length'] > 3000])
print(f"Number of rows with doc_length > 3000: {num_rows_greater_than_3000}")

num_rows_200_to_1000 = len(df[(df['doc_length'] > 200) & (df['doc_length'] < 1000)])
print(f"Number of rows with doc_length > 200 and < 1000: {num_rows_200_to_1000}")

Number of rows with doc_length < 300: 221
Number of rows with doc_length > 3000: 78
Number of rows with doc_length > 200 and < 1000: 100
